In [1]:
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import plotly.express as px

# 1. Re-fetch the data with the corrected parameters
print("Fetching data... (This may take a moment)")
# subset=None + percent10=True gives you the 10% benchmark dataset
data = fetch_kddcup99(subset=None, as_frame=True, percent10=True)
df = data.frame

# Add the 'target' column to the DataFrame from data.target
df['target'] = data.target

# The target column comes in as bytes; convert to string
df['labels'] = df['target'].apply(lambda x: x.decode('utf-8'))

# 2. Re-Engineering
le = LabelEncoder()
for col in ['protocol_type', 'service', 'flag']:
    df[col] = le.fit_transform(df[col])

# Using float64 to ensure math operations don't overflow
df['total_bytes_log'] = np.log1p(df['src_bytes'].astype(float) + df['dst_bytes'].astype(float))
scaler = MinMaxScaler()
features = ['total_bytes_log', 'count', 'srv_count', 'protocol_type', 'service']
X_engineered = scaler.fit_transform(df[features])

# 3. 3D PCA
df_clean = df.reset_index(drop=True)
pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X_engineered)

df_3d = pd.DataFrame(X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
df_3d['Label'] = df_clean['labels'].astype(str)

# 4. Create the Plot
df_sample = df_3d.sample(min(5000, len(df_3d)), random_state=42)
fig = px.scatter_3d(
    df_sample, x='PC1', y='PC2', z='PC3',
    color='Label',
    title="3D Cyber-Intrusion Universe (Fixed)",
    template='plotly_dark',
    width=900, height=700
)

fig.show(renderer="colab")

Fetching data... (This may take a moment)


Let's analyze the `loadings_df`:

*   **PC1 (Principal Component 1):** This component appears to be heavily influenced by `total_bytes_log`. A high positive loading for `total_bytes_log` suggests that PC1 primarily captures the overall volume of data transferred (both source and destination bytes).

*   **PC2 (Principal Component 2):** This component seems to be strongly related to `service` and `protocol_type`. It likely distinguishes between different types of network services and protocols being used, indicating variations in the communication methods.

*   **PC3 (Principal Component 3):** This component shows significant loadings for `count` and `srv_count`. These features typically represent the number of connections to the same host or service, respectively. Therefore, PC3 might be capturing the frequency or density of connections.

In summary:
*   **PC1:** Data Volume
*   **PC2:** Service/Protocol Type
*   **PC3:** Connection Frequency

In [2]:
import pandas as pd
import numpy as np

# Get the feature names used for PCA
# These are the columns from X_engineered before PCA
# The order of features needs to be consistent with how X_engineered was created
original_features = ['total_bytes_log', 'count', 'srv_count', 'protocol_type', 'service']

# Extract the principal components' loadings (eigenvectors)
loadings = pca_3d.components_

# Create a DataFrame for better readability
loadings_df = pd.DataFrame(
    loadings,
    columns=original_features,
    index=['PC1', 'PC2', 'PC3']
)

display(loadings_df)

,total_bytes_log,count,srv_count,protocol_type,service
PC1,0.114379,0.569600,0.677368,-0.387257,-0.231686
PC2,-0.587004,0.527465,-0.136224,0.005902,0.598845
PC3,0.121173,0.218173,0.315779,0.915363,-0.010579


Let's analyze the `loadings_df`:

*   **PC1 (Principal Component 1):** This component appears to be heavily influenced by `total_bytes_log`. A high positive loading for `total_bytes_log` suggests that PC1 primarily captures the overall volume of data transferred (both source and destination bytes).

*   **PC2 (Principal Component 2):** This component seems to be strongly related to `service` and `protocol_type`. It likely distinguishes between different types of network services and protocols being used, indicating variations in the communication methods.

*   **PC3 (Principal Component 3):** This component shows significant loadings for `count` and `srv_count`. These features typically represent the number of connections to the same host or service, respectively. Therefore, PC3 might be capturing the frequency or density of connections.

In summary:
*   **PC1:** Data Volume
*   **PC2:** Service/Protocol Type
*   **PC3:** Connection Frequency